# Extract

Lectura del dataset crudo, validación de estructura y calidad básica, y exportación del raw data limpio para la etapa de transformación.

---

In [1]:
import pandas as pd
import numpy as np

## 1. Lectura del dataset

In [ ]:
df = pd.read_csv('../data/raw/Dataset DS - 26.csv')
print('Shape:', df.shape)
df.head()

In [3]:
print('Tipos de datos:')
print(df.dtypes)

Tipos de datos:
user_id                   object
install_time              object
platform                  object
country_region            object
city                      object
gender                    object
min_age_range              int64
max_age_range              int64
event_1                    int64
event_2                    int64
event_3                    int64
event_4                    int64
event_5                    int64
target_churn_indicator     int64
dtype: object


## 2. Validación

### Nulos

In [4]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
pd.DataFrame({'nulos': nulos, '% nulos': nulos_pct})[nulos > 0]

,nulos,% nulos
country_region,479,2.40
city,935,4.68


Los nulos se concentran en variables geográficas (`country_region` 2.4%, `city` 4.68%). Se tratan en la etapa de transformación imputando `"unknown"`.

### Duplicados

In [5]:
n_dupes = df['user_id'].duplicated().sum()
print(f'user_id duplicados: {n_dupes} ({n_dupes/len(df)*100:.1f}%)')
print(f'user_id únicos: {df["user_id"].nunique()}')

user_id duplicados: 1467 (7.3%)
user_id únicos: 18533


In [6]:
dupes = df[df['user_id'].duplicated(keep=False)]

target_conflict = dupes.groupby('user_id')['target_churn_indicator'].nunique()
time_conflict = dupes.groupby('user_id')['install_time'].nunique()

print('Duplicados con target distinto:', (target_conflict > 1).sum())
print('Duplicados con install_time distinto:', (time_conflict > 1).sum())

Duplicados con target distinto: 0
Duplicados con install_time distinto: 0


Los duplicados comparten `install_time` y `target_churn_indicator` pero difieren en geolocalización y conteo de eventos — ruido del pipeline de datos, no reinstalaciones. Se elimina la segunda ocurrencia.

## 3. Limpieza

In [7]:
df = df.drop_duplicates(subset='user_id', keep='first').reset_index(drop=True)
print('Shape final:', df.shape)

Shape final: (18533, 14)


## 4. Export

In [ ]:
df.to_parquet('../data/raw/data_raw.parquet', index=False)
print('Guardado en data/raw/data_raw.parquet')